<img src="../logo.png" alt="Vegeta" width="240">

# Kestrel-X — Kestrel's motors, battery and pod on four cantilever panels, no bracing

**Kestrel-X** is the X-wing sibling of the Kestrel-4Q biplane, a study of its own: the same four motors, battery and pod on four straight panels at ±45°, without the biplane's truss. It is compared with **Kestrel-4Q** as notebook `43_kestrel_study` last exported it (`designs/data/reports/43_kestrel_study.json`). When that report is missing the notebook says so loudly and goes on with Kestrel-X alone; it never invents the biplane's numbers. (It grew out of `old/41_xwing`, which read notebook 40's older record.)

The biplane of notebook 40 carries its four rotors on two wings joined by a truss. The **X-wing** carries the same
four rotors, the same battery and the same pod on **four straight cantilever panels radiating at ±45°** — seen as
wings, two straight wings of equal span crossing at the pod, one rolled +45° and one −45°. There is no truss to weigh,
to fair or to brace; the aircraft stands on its four tips. The question: with everything bought held equal, does the
X beat the braced biplane — and what does the 45° cost?

The geometry decides a lot before any analysis: a panel's lift is normal to the panel, so only cos 45° of it is
vertical, and the panel sees only cos 45° of the aircraft's angle of attack. The vertical lift slope of four panels is
about half that of the same area laid flat. Two variants: **X-A** with the biplane's wing area (4 × 0.50 m × 0.14 m) and
**X-B** with twice it (4 × 0.70 m × 0.20 m).

```
layout (kestrel_xwing) ─► clearances, the motors' stations (pitch and yaw levers equal), projected span
masses: the biplane's parts without the truss and the fin, the spars sized as cantilevers (stress, deflection, first mode)
hold point: the allocation matrix on the X's stations (kestrel_control with L=)
lattice (peregrine_flight.vlm) on four panels at 45° ─► vertical lift slope, induced drag, neutral point
drag build-up (no struts), cruise with the pitch trimmed by the rotors' vertical lever, stance on the tips
transitions in notebook 40's panel model (each panel at its dihedral, blown by its own rotor): the matrix, the strategies
against Kestrel-4Q and Kestrel-4Q-2 (designs/data/kestrel_design.json)
```
Modules: `designs/kestrel_xwing.py` (new), `kestrel_transition.py` (the panel model), `kestrel_control.py`,
`kestrel_systems.py`, `kestrel_drawings.py`. **Status: concept study**, the same assumptions as notebook 40.

In [ ]:
# DEBUG — ON BY DEFAULT: the expensive solvers are mocked (vegeta.mock: fixed numbers, seconds instead of hours,
# labelled MOCK). For real results set DEBUG = False here, or from the shell: VEGETA_DEBUG=0 jupyter nbconvert …
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)        # the lab's folders on sys.path, the mocks

In [ ]:
import json, math, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0, ".")
import kestrel, kestrel_systems as ks, kestrel_control as kc, kestrel_flight as kf, kestrel_transition as kt, kestrel_xwing as kx, kestrel_drawings as kd

pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 220)
OUT = Path("../output/45_kestrel_x"); OUT.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("XWING_QUICK") == "1"
import yanus                                       # Ozaru's requirement checks: what this notebook needs on this machine
from yanus import Package
NEEDS = [Package("vegeta.boreas"),
         Package("vegeta.dedalus")]
print(yanus.table(yanus.check_needs(NEEDS)) if NEEDS else "needs: nothing beyond the Python environment")
BIP = kx.biplane_reference()                       # notebook 43's report (designs/data/reports/43_kestrel_study.json); None, with a loud warning, when it is missing
print("Kestrel-4Q (the biplane):", f"notebook 43's report, {BIP['source']}" if BIP else "NOT FOUND — the comparison rows are left out (see the warning above)")
display(pd.DataFrame({v: p for v, p in kx.VARIANTS.items()}).T)

## 1. The two X-wings

The layout, its rules (the discs clear each other, the pod and the tips) and the front views. The motors sit at 45°:
the pitch lever (their height) equals the yaw lever (their spanwise station) — 0.33 m for X-A against the biplane's
0.14 m pitch and 0.39 m yaw levers.

In [ ]:
XL = {v: kx.layout(v) for v in kx.VARIANTS}
fig, axs = plt.subplots(1, 2, figsize=(14, 6))
for ax_, (v, L) in zip(axs, XL.items()):
    kd.xwing_front_view(L, ax=ax_)
plt.show()
LAY = pd.DataFrame({v: {k: L[k] for k in ("panel_length", "chord", "motor_r", "span", "span_projected", "height", "S_ref", "AR_wing", "h", "motor_y",
                                           "prop_gap", "prop_to_pod", "prop_beyond_tip", "elevon_y_in", "elevon_span_side")} for v, L in XL.items()}).T
display(LAY.round(3))
print({v: kx.checks(L) or "all rules hold" for v, L in XL.items()})

### 1b. The two X-wings in 3-D

Drawn from the layout (the X has no CAD yet, todo Y.9): the four panels at ±45° with their elevons shaded, the pod, the
propeller discs at the panels' tips, the foot rods the aircraft stands on. Change `elev` / `azim` to turn it.

In [ ]:
fig = plt.figure(figsize=(16, 7))
for k, (v, L) in enumerate(XL.items(), start=1):
    ax3 = fig.add_subplot(1, 2, k, projection="3d")
    kd.xwing_3d(L, ax=ax3, elev=22, azim=-55)
plt.tight_layout(); plt.show()
fig = plt.figure(figsize=(8, 7)); kd.xwing_3d(XL[list(kx.VARIANTS)[0]], ax=fig.add_subplot(111, projection="3d"), elev=8, azim=-90); plt.show()   # from the left side

## 2. Masses and the spars

The biplane's parts list without the truss and the fin plate (the panels are the fins); each panel's spar sized as a
cantilever with a tip motor at a 4 g pull-up — on stress (≤ 200 MPa), deflection (≤ 3 % of the length) and the first
bending mode of the motor on its arm (≥ 12 Hz, an octave above the pitch loop); the elevon servos; the feet at the
tips. The battery placed for the same CG rule as Kestrel's.

In [ ]:
XT = {v: kx.mass_table(L) for v, L in XL.items()}
XCI = {v: ks.cg_inertia(t) for v, t in XT.items()}
MASS = pd.DataFrame({v: ks.groups(t) for v, t in XT.items()})
display(MASS.round(3))
display(pd.DataFrame({v: {"take-off mass [kg]": c["mass_kg"], "CG x [mm]": c["x_cg"] * 1000, "battery x [mm]": c["battery_x"] * 1000, "clipped": c["battery_clipped"],
                          "Ixx": c["Ixx"], "Iyy": c["Iyy"], "Izz": c["Izz"]} for v, c in XCI.items()}).T.round(4))
SPAR = pd.DataFrame({v: kx.cantilever(L, mass_kg=XCI[v]["mass_kg"]) for v, L in XL.items()}).T
display(SPAR)
print("Kestrel-4Q, take-off mass (notebook 43):", {k: f"{v['take-off mass [kg]']:.3f} kg" for k, v in BIP["variants"].items()} if BIP else "— not found")

## 3. The hold point

The same allocation matrix on the X's stations: the senses of rotation, the attainable moments, a motor out.

In [ ]:
DR = ks.drive_table()
for v, L in XL.items():
    ci = XCI[v]
    print(v)
    display(kc.rank_table(kappa=float(DR.kap(0.0)), cg=ci, L=L)[["senses (M1..M4)", "rank", "condition", "verdict"]])
    display(kc.hover_authority(mass_kg=ci["mass_kg"], cg=ci, inertia=ci["I"], L=L).round(3))

## 4. Aerodynamics: the lattice at 45°, the drag, the cruise

The vortex lattice on the four panels as quads at ±45°: the vertical lift slope on the wing area (against the flat
biplane's), the induced-drag factor on the projected span, the neutral point. The drag build-up has no struts. The
cruise: the vertical CL the weight needs, the panels' own lift coefficient (CL / cos 45°), the angle of attack, the
power with the pitch trimmed by the rotors' vertical lever.

In [ ]:
XA = {v: kx.aero(L) for v, L in XL.items()}
XD = {v: kx.drag_buildup(L) for v, L in XL.items()}
BIPLANE = kf.aero()["biplane"]                      # notebook 40's lattice on the flat pair, for the ratio
AER = pd.DataFrame({v: {"CLα vertical [1/rad]": a["CLa"], "CLα / the biplane's": a["CLa"] / BIPLANE["CLa"],
                        "induced k": a["k"], "e (projected span)": a["e"], "AR projected": a["AR_projected"], "x_np [mm]": a["x_np"] * 1000,
                        "Cd0": XD[v]["Cd0"], "CdA [cm²]": XD[v]["CdA_m2"] * 1e4} for v, a in XA.items()}).T
display(AER.round(4))
display(pd.DataFrame({v: {k: x * 1e4 for k, x in d["parts_m2"].items()} for v, d in XD.items()}).T.round(1).rename(columns=lambda c: c + " [cm²]"))
XV = {v: kx.make_vehicle(v) for v in kx.VARIANTS}
CRX = {v: kx.cruise(veh) for v, veh in XV.items()}
for v, c in CRX.items():
    print(v); display(c.round(3))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for v, c in CRX.items():
    f = c[c["feasible"]]
    ax[0].plot(f.index, f["electrical [W]"], "o-", label=v); ax[1].plot(f.index, f["endurance [min]"], "o-", label=v)
for a_ in ax:
    a_.set_xlabel("airspeed [m/s]"); a_.legend(fontsize=8)
ax[0].set(ylabel="W", title="electrical power in level flight"); ax[1].set(ylabel="min", title="endurance (80 % of the battery)")
plt.show()

### 4b. The panels' section (Zephyr)

What the wing section sees. A panel at 45° must carry its lift at cl = CL_vertical / cos 45°: X-A's panels run at
cl ≈ 0.50 at 16 m/s where the biplane's wings run at 0.38, X-B's at 0.30. From Zephyr's NACA 0012 polar (NeuralFoil,
Re 1.5e5, n_crit 6) and its flap sweep: the section drag at that cl, the elevon that trims the panel with the X's own
static margin (the lattice's neutral point against the CG), that elevon's drag, the section L/D with the trim in,
the margin to cl_max — against the two Kestrels; then over the speed range, and which reflexed section would trim the
X with its elevons level.

In [ ]:
try:
    import kestrel_section as ksec
    SECT = ksec.table()
    HAVE_ZEPHYR = True
except Exception as exc:
    HAVE_ZEPHYR = False
    print("Zephyr / the section data not available:", type(exc).__name__, exc)
if HAVE_ZEPHYR:
    def section_row(sec, cl, sm):
        p = sec["polar_cruise"]; cl_ = np.array(p["cl"]); i = int(np.argmax(cl_))
        at = lambda key: float(np.interp(cl, cl_[:i + 1], np.array(p[key])[:i + 1]))
        cd, cm = at("cd"), at("cm")
        e = sec["elevon"]
        d_deg = math.degrees((sm * cl - cm) / e["dcm_ddelta_per_rad"])
        dcd = float(np.interp(d_deg, e["deltas_deg"], e["cd_increment"]))
        return {"section cl": cl, "section cd": cd, "trim elevon [deg]": d_deg, "Δcd of the trim": dcd, "cd with the trim": cd + dcd,
                "section L/D (trimmed)": cl / (cd + dcd), "cl max": sec["cruise"]["cl_max"], "margin to stall": sec["cruise"]["cl_max"] / cl,
                "cl lost to the elevon": e["dcl_ddelta_per_rad"] * math.radians(d_deg)}
    S0012 = SECT["NACA 0012"]
    SMX = {v: (XA[v]["x_np"] - XCI[v]["x_cg"]) / XL[v]["chord"] for v in kx.VARIANTS}
    rows = {}
    for v in kx.VARIANTS:
        c16 = CRX[v].loc[16.0]
        rows[f"{v[:3]} (0012, panels)"] = section_row(S0012, float(c16["panel cl"]), SMX[v])
    sm_k = float(BIP["variants"]["Kestrel-4Q-1"]["static margin [chords]"]) if BIP and "Kestrel-4Q-1" in BIP["variants"] else 0.103
    rows["Kestrel-4Q (0012)"] = section_row(S0012, 0.38, sm_k)
    pick = BIP["variants"][BIP["main"]]["section"].replace(" (Zephyr)", "") if BIP and BIP["main"] else "NACA 23112 (k2/k1 0.03)"
    if pick in SECT:
        rows[f"Kestrel-4Q-2 ({pick})"] = section_row(SECT[pick], 0.38, sm_k)
    SEC16 = pd.DataFrame(rows).T
    print("the X's static margins (lattice neutral point − CG, in chords):", {v[:3]: round(s, 3) for v, s in SMX.items()})
    display(SEC16.round(4))
    # over the speed range: the panels' cl and the trimmed section L/D
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    for v in kx.VARIANTS:
        c = CRX[v][CRX[v]["feasible"]]
        rr = [section_row(S0012, float(cl), SMX[v]) for cl in c["panel cl"]]
        ax[0].plot(c.index, c["panel cl"], "o-", label=v); ax[1].plot(c.index, [r["section L/D (trimmed)"] for r in rr], "o-", label=v)
    ax[0].axhline(S0012["cruise"]["cl_max"], color="k", ls=":", lw=1, label="cl max (0012, Re 1.5e5)")
    ax[0].set(xlabel="airspeed [m/s]", ylabel="panel cl", title="the panels' lift coefficient"); ax[1].set(xlabel="airspeed [m/s]", ylabel="cl / cd", title="the section's L/D with the trim elevon in")
    ax[0].legend(fontsize=8); ax[1].legend(fontsize=8); plt.show()
    # which section trims the X with the elevons level: the trim table at the X's margins and panel cl
    for v in kx.VARIANTS:
        print(v)
        display(ksec.trim_table(SECT, static_margin=SMX[v], cl=float(CRX[v].loc[16.0, "panel cl"]))[["cm needed", "cm of the section", "elevon [deg]", "Δcd (elevon)", "cd cruise", "trim drag / section drag"]].astype(float).round(4))

### 4c. The medical capsule in the X's pod

Kestrel carries a removable insulated capsule for medical supplies (72 × 72 × 160 mm, 0.6 L, up to 0.5 kg;
notebook 40 § 11). The biplane stacks the battery, the capsule and the electronics in the 274 mm of pod height
between its wings. The X's pod spans no gap — it is 110 mm square — so the same parts must lie end to end.

In [ ]:
CAPX = pd.DataFrame({v: {k: x for k, x in kx.capsule_fit(L).items() if k != "note"} for v, L in XL.items()}).T
display(CAPX)
print(kx.capsule_fit(XL[list(kx.VARIANTS)[0]])["note"])
m_cap = ks.COMPONENTS["capsule"]["shell_kg"] + ks.COMPONENTS["capsule"]["contents_nominal_kg"]
print({v: f"with the nominal capsule: {XCI[v]['mass_kg'] + m_cap:.2f} kg, hover T/W {ks.hover_point(XCI[v]['mass_kg'] + m_cap)['T_W']:.2f}" for v in kx.VARIANTS})

## 5. Standing on the tips

The footprint is the square of the four tips; the wind that tips it (the panels' projected area).

In [ ]:
ST = pd.DataFrame({v: {k: x for k, x in kx.stance(L, mass_kg=XCI[v]["mass_kg"], x_cg=XCI[v]["x_cg"]).items() if k != "feet"} for v, L in XL.items()}).T
display(ST.round(2))

## 6. The transitions

Notebook 40's panel model with the X's four panels at 45°: each sees cos 45° of the vertical flow and gives back cos 45°
of its normal force; each is blown by its own rotor; the autopilot trims with the X's own vertical lift slope. The
matrix (both directions, three schedule lengths, still air and a 6 m/s headwind), the nominal episodes, and the back
transition entered slowly.

In [ ]:
durs = (1.5, 3.0) if QUICK else (1.5, 3.0, 5.0)
MATX, EPSX = kt.matrix(XV, durations=durs, headwinds=(0.0, 6.0), progress=True)
cols = ["height loss [m]", "height gain [m]", "pitch error max [deg]", "time stalled, free stream [s]", "α blown max while stalled [deg]",
        "moment shortfall [s]", "least pitch margin", "peak rotor thrust / T_max", "elevon max [deg]", "roll demand / authority",
        "roll demand / rotors' authority", "pass"]
display(MATX[cols].astype({c: float for c in cols if c != "pass"}).round(2))
VERX = kt.verdict(MATX)
for k, v in VERX.items():
    print(f"{k}: {v['passes']} of {v['of']} pass" + ("".join(f"\n   - {w}" for w in v["failures"]) if v["failures"] else ""))
MATX.to_csv(OUT / "transition_matrix.csv")
SLOW = pd.DataFrame({f"{v[:3]}, back entered at {ve:g} m/s": kt.simulate(veh, "back", t_tr=3.0, V_entry=ve).metrics for v, veh in XV.items() for ve in (16.0, 13.0, 11.0)}).T
display(SLOW[cols])
for v, veh in XV.items():
    kd.episode_figure(EPSX[(v, "forward", 3.0, 0.0)], veh); plt.show()
    kd.episode_figure(EPSX[(v, "back", 3.0, 0.0)], veh); plt.show()

## 7. Against Kestrel-4Q (the biplane, notebook 43)

In [ ]:
rows = {}
for v in kx.VARIANTS:
    L, veh, c = XL[v], XV[v], CRX[v]
    feas = c[c["feasible"]]; c16 = c.loc[16.0]
    hx = kc.hover_authority(mass_kg=veh.mass, cg={"x_cg": veh.x_cg, "y_cg": 0.0, "z_cg": veh.z_cg}, inertia=XCI[v]["I"], L=L)
    rows[v] = {"mass [kg]": veh.mass, "wing area [m²]": L["S_ref"], "projected span [m]": L["span_projected"], "CLα vertical": XA[v]["CLa"], "induced k": XA[v]["k"],
               "Cd0": veh.pol.cd0, "CdA [cm²]": veh.pol.cd0 * L["S_ref"] * 1e4, "α at 16 m/s [deg]": c16.get("α [deg]", np.nan), "panel cl at 16 m/s": c16["panel cl"],
               "power at 16 m/s [W]": c16.get("electrical [W]", np.nan), "endurance best [min]": feas["endurance [min]"].max(), "at [m/s]": feas["endurance [min]"].idxmax(),
               "hover pitch [rad/s²]": hx.iloc[1]["angular acceleration [rad/s²]"], "hover heading [rad/s²]": hx.iloc[0]["angular acceleration [rad/s²]"],
               "spar OD [mm]": SPAR.loc[v, "spar OD [mm]"], "tip-over [deg]": ST.loc[v, "least tip-over angle [deg]"], "wind that tips it [m/s]": ST.loc[v, "wind that tips it [m/s]"],
               "transitions passed": f"{VERX[v]['passes']} of {VERX[v]['of']}", "bracing": "none"}
if BIP:                                             # Kestrel-4Q as notebook 43 exported it (its lattice: CLα and k of the flat pair)
    for name, r in BIP["variants"].items():
        rows[name] = {"mass [kg]": r["take-off mass [kg]"], "wing area [m²]": 0.28, "projected span [m]": 1.0, "CLα vertical": BIPLANE["CLa"], "induced k": BIPLANE["k"],
                      "Cd0": r["Cd0"], "CdA [cm²]": r["Cd0"] * 0.28 * 1e4, "power at 16 m/s [W]": r["power at 16 m/s [W]"], "endurance best [min]": r["best endurance [min]"],
                      "transitions passed": r["transitions passed"], "bracing": f"{r['bracing']}, {r['struts']}"}
    main = rows.get(BIP["main"])
    if main is not None:
        g = BIP["ground"]
        main.update({"hover pitch [rad/s²]": BIP["hover pitch [rad/s²]"], "hover heading [rad/s²]": BIP["hover heading [rad/s²]"],
                     "tip-over [deg]": g.get("least tip-over angle [deg]"), "wind that tips it [m/s]": g.get("wind that tips it, broadside [m/s]")})
else:
    kx.warn_missing("Kestrel-4Q's rows for this table", "notebook 43's report was not found", "run notebook 43_kestrel_study, then this notebook again")
CMP = pd.DataFrame(rows).T
display(CMP)
CMP.to_csv(OUT / "kestrel_x_vs_4q.csv")

### 7b. Drag, power, maximum speed and manoeuvrability against airspeed — all four

Kestrel-4Q-1 and 4Q-2 rebuilt here from the same modules (4Q-2 as elevons + faired posts, no X; the reflexed section's
small change is left out of this comparison), the two X-wings as above. The drag split into parasite and induced, the
rotors' full thrust against the drag (the thrust-limited maximum speed), the power; then pitch, roll, yaw and the turn
in horizontal flight.

In [ ]:
P2 = kestrel.resolve(dict(elevons=True, strut_fairing=True, bracing="posts"))
FOUR = {"Kestrel-4Q-1 (round struts, X, no elevons)": kt.make_vehicle(elevons=False, name="4Q-1"),
        "Kestrel-4Q-2 (elevons, faired posts, no X)": kt.make_vehicle(P2, faired=True, name="4Q-2")}
FOUR.update({v[:3]: XV[v] for v in kx.VARIANTS})
fig, axs = plt.subplots(1, 3, figsize=(17, 4.8))
DRAG4 = kd.drag_vs_speed(FOUR, speeds=np.arange(8.0, 40.5, 0.5), axs=axs); plt.tight_layout(); plt.show()
display(DRAG4.round(2))
fig, axs = plt.subplots(2, 2, figsize=(15, 9))
MANV4 = kd.manoeuvrability_figure(FOUR, axs=axs); plt.tight_layout(); plt.show()
for k, df in MANV4.items():
    print(k); display(df.loc[[v for v in (14.0, 16.0, 20.0, 25.0) if v in df.index]].round(2))
DRAG4.to_csv(OUT / "drag_vs_speed.csv")

## 8. Findings

**Lift.** A panel at 45° gives back cos 45° of its normal force vertically and sees cos 45° of the vertical flow: the
lattice puts the X's vertical lift slope at **1.88 /rad against the biplane's 4.08** (0.46 ×) with the same area and
motors. So X-A (the biplane's 0.28 m²) flies 16 m/s at 10.8° with its panels at cl 0.50 and stalls at 12.6 m/s —
**1.27 × the stall** where the biplane has 1.47 ×; X-B buys its margin (stall 9.7 m/s, 6.4° at the cruise) with twice
the area, not better geometry — 0.29 kg of spars and panels more (1.88 kg against X-A's 1.59 and the biplane's 1.65–1.69).

**Drag, power, speed.** No truss and no fin: X-A's Cd0 0.034 (96 cm²) sits between the round-strut biplane (0.067,
186 cm²) and the faired 4Q-2 (0.026, 72 cm²); X-B's 0.030 is on twice the area (170 cm²). At 16 m/s: **X-A 88 W, X-B
110 W, 4Q 110 W, 4Q-2 59 W**; the best endurance 29 min at 15 m/s, 33 at 12, 40, **57**. The thrust-limited maximum
speed is within 3 m/s for all four (30–33.5 m/s): the 9×6's pitch speed, not the airframe, sets it.

**Transitions.** In the same panel model X-A passes **12 of 12** at the cruise entry and X-B **10 of 12** (the 1.5 s
back transition fails on roll: the big 45° panels' one-sided stall against the rotors' torque). The X's rule is the
biplane's inverted: the X must enter the back transition **fast** — X-A entered at 13 m/s (1.03 × its stall) sinks
4.3 m, at 11 m/s 6.1 m — where 4Q-2 must enter slowly (≤ 13 m/s) or balloon. In cruise the elevons give X-A the
biplane's roll rate (350°/s against 4Q-2's 400) but a third of its pitch (16 against 49 rad/s²), and at the lift
limit X-A turns at 45°/s in a 21 m radius (4Q-2: 69°/s, 13 m; X-B, on its area: 89°/s, 10 m).

**The hover and the ground.** Pitch authority 67 rad/s² (X-A) and 51 (X-B) against the biplane's 95; the heading
1.8 and 0.9 against 2.35 — the same weak axis. Standing on its four tips the X tips over at 60° and in a 16 m/s wind
(the biplane: 39°, 9 m/s broadside): the better landing gear by far.

**The section (Zephyr).** With NACA 0012 and a 0.134 static margin X-A's trim costs −9.3° of elevon and half its
section lift at the cruise cl of 0.50 (X-B: −5.6°); the reflexed NACA 23112 (k₂/k₁ 0.03) trims X-A at −3.5° and
X-B at −0.7°, the same pick as the biplane's for the same reason.

**The medical capsule.** The 72 × 72 × 160 mm capsule fits the X's pod section (98 mm inside the walls, 13 mm to
spare) but not in line with the battery and the electronics: 462 mm needed in a 430 mm pod. The biplane stacks the
three in 274 mm of pod height; the X must lengthen its pod by ~40 mm or carry the contents far from the CG. Carried,
the nominal capsule puts X-A at 1.98 kg (hover T/W 2.47) and X-B at 2.27 kg (2.15).

**Which one for what.** **X-A** is the simplest, lightest and most robust airframe here — no truss, no fin, four spars,
the best stance on the ground, every transition passed at the cruise entry — for short sorties that are mostly hover
and a small payload; it pays with a cruise 1.27 × its stall, 50 % more power than 4Q-2 and a pod that needs
lengthening for the capsule. **X-B** buys the stall margin and the tightest turn with area, and gives up the X's
lightness (1.88 kg, 110 W) without reaching the biplane's endurance; it is the one to pass over. **Kestrel-4Q-2** is
the aircraft for the medical mission — 57 min against 29–33, 59 W against 88–110, the capsule stacked on the axis,
the strongest pitch in the hover — on two conditions it must keep: the faired, braced truss and the slow entry into
the back transition. Every number is the lattice, Boreas and the panel model; the X-wing has no CFD yet (notebook 39
needs a CAD class for it first) and its roll–yaw coupling through 45° panels waits for the 6-DOF run.

In [ ]:
def plain(x):
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    return x
rows_ = lambda df: {" | ".join(map(str, k)) if isinstance(k, tuple) else str(k): {kk: plain(vv) for kk, vv in r.items()} for k, r in df.T.to_dict().items()}
DESIGN = {"source": "45_kestrel_x", "biplane_reference": BIP["source"] if BIP else None, "variants": kx.VARIANTS, "layout": {v: {k: plain(x) for k, x in L.items() if isinstance(x, (int, float))} for v, L in XL.items()},
          "mass": {v: {k: plain(x) for k, x in c.items() if k != "I"} for v, c in XCI.items()}, "aero": {v: {k: plain(x) for k, x in a.items() if k != "note"} for v, a in XA.items()},
          "spars": rows_(SPAR), "transitions": rows_(MATX[cols]), "verdict": VERX, "slow_entry": rows_(SLOW[cols]), "comparison": rows_(CMP.astype(object)),
          "drag_vs_speed": rows_(DRAG4), "manoeuvrability_at_16": {k: {kk: plain(vv) for kk, vv in df.loc[16.0].items()} for k, df in MANV4.items() if 16.0 in df.index}}
target = OUT / "kestrel_x_design.json" if DEBUG else Path("../designs/data/kestrel_x_design.json")
target.write_text(json.dumps(DESIGN, indent=1, default=str))
print("written", target)

## 9. The dossier

This notebook's numbers as JSON (`designs/data/reports/45_kestrel_x.json`) and Markdown, stitched into `designs/data/kestrel_dossier.json` / `.md`.

In [ ]:
# ---- the design dossier: every number of this notebook as Markdown to paste into a conversation (kestrel_report)
import kestrel_report as kr
def _g(n):
    return globals().get(n)
def _chk(names):
    import pandas as pd
    out = {}
    for n in names:
        c = _g(n)
        if c:
            out[n] = kl.check_table(c, n)
    return out

import kestrel_loop as kl
REPORT = {"variants": pd.DataFrame(kx.VARIANTS).T, "layout": _g("LAY"), "masses": _g("MASS"), "spars": _g("SPAR"), "hold point": _g("AER"),
          "aerodynamics": pd.DataFrame({v: {k: x for k, x in a.items() if k != "note"} for v, a in XA.items()}).T, "cruise tables": {v: df.loc[[s for s in (12.0, 16.0, 20.0) if s in df.index]] for v, df in CRX.items()} if isinstance(_g("CRX"), dict) else _g("CRX"),
          "section (Zephyr)": _g("SEC16") if _g("SEC16") is not None else _g("SECX"), "capsule": _g("CAPX"), "standing on the tips": _g("ST"),
          "transitions": MATX[cols] if _g("MATX") is not None else None, "verdict": VERX, "back entry speeds": SLOW[cols] if _g("SLOW") is not None else None,
          "against Kestrel-4Q (notebook 43)": CMP, "drag vs speed, all four": DRAG4, "manoeuvrability at 16 m/s, all four": pd.DataFrame({k: df.loc[16.0] for k, df in MANV4.items() if 16.0 in df.index}).T}
REPORT = {k: v for k, v in REPORT.items() if v is not None}
EXP = kr.export("45_kestrel_x", "Kestrel-X — notebook 45 (X-A and X-B against Kestrel-4Q, notebook 43)", REPORT)
print("JSON report:", EXP["json"], f"({EXP['json_kB']} kB) | JSON dossier:", EXP["dossier_json"], f"({EXP['dossier_json_kB']} kB) | Markdown:", EXP["markdown"])
kr.paste("45_kestrel_x", max_lines=30)